# KYRO CLOUD — Colab Runtime Agent

Run the cells top-to-bottom. This installs the desktop + streaming stack, then starts the agent that connects to the Render backend.

After the last cell shows the agent connected, open **https://kyro-cloud.vercel.app**, log in, and click **Start Cloud PC**.

**Do not stop the notebook runtime** while you want the cloud PC online.

In [ ]:
import os

REPO = "/content/kyro-cloud"
if os.path.isdir(REPO):
    print("Repo exists, pulling latest...")
    !cd {REPO} && git pull
else:
    print("Cloning repo...")
    !git clone https://github.com/harshpreetsaini/kyro-cloud {REPO}
%cd {REPO}
!git log --oneline -1

In [ ]:
# --- Runtime secrets (must match Render's RUNTIME_AUTH_SECRET) ---
os.environ["LUNA_BACKEND_WS"]      = "wss://kyro-cloud-3fp0.onrender.com/agent"
os.environ["RUNTIME_AUTH_SECRET"]  = "f5bc62648888d9ae066231b0535eb0643fdce193991581df541b19562c9c2a796"
os.environ["LUNA_DISPLAY"]         = ":1"
print("Environment configured:")
print("  LUNA_BACKEND_WS     =", os.environ["LUNA_BACKEND_WS"])
print("  RUNTIME_AUTH_SECRET =", os.environ["RUNTIME_AUTH_SECRET"][:8] + "...")
print("  LUNA_DISPLAY        =", os.environ["LUNA_DISPLAY"])

In [ ]:
# Install desktop + VNC + apps, then start the agent (background).
# We use subprocess to pass the Python env vars to the shell.
import subprocess
env = os.environ.copy()
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
result = subprocess.run(
    ['bash', 'runtime-agent/bootstrap/bootstrap.sh'],
    env=env, cwd=REPO
)
if result.returncode != 0:
    print(f'WARNING: bootstrap exited with code {result.returncode}')

In [ ]:
# Give the agent a few seconds, then show its log.
# Look for: "[agent] connected" / "ready" and NO auth errors.
import time
time.sleep(6)
!tail -n 40 /tmp/luna-agent.log

In [ ]:
# Verify the backend is reachable from Colab.
import urllib.request, json
try:
    with urllib.request.urlopen("https://kyro-cloud-3fp0.onrender.com/api/health", timeout=15) as r:
        print("Backend health:", json.loads(r.read()))
except Exception as e:
    print("Backend health check failed:", e)

## Done

If the log shows the agent connected and the backend health returned `kyro-cloud-backend`, go to **https://kyro-cloud.vercel.app**, log in, and click **Start Cloud PC**.

State flow: `OFFLINE → STARTING → RUNTIME_CONNECTED → GPU_READY → DESKTOP_READY → STREAM_STARTING → STREAMING → ONLINE`.

To stop the agent (e.g. before closing the notebook):

In [ ]:
!pkill -f "python3 main.py" 2>/dev/null && echo "Agent stopped." || echo "No agent running."